----------------------------------------------------------------------------------------------
--------------------------------- ROW REMOVAL (NEIGHBORING DATASETS) -------------------------
----------------------------------------------------------------------------------------------



In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
import copy
import os

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


/Users/elif/Desktop/Projects/privacy_in_ml


In [2]:
diabetes_train = pd.read_csv("datasets/diabetes_train.csv")
diabetes_test = pd.read_csv("datasets/diabetes_test.csv")

X_train = diabetes_train.drop('Outcome', axis=1)
X_test = diabetes_test.drop('Outcome', axis=1)
y_train = diabetes_train['Outcome']
y_test = diabetes_test['Outcome']


In [3]:
def preprocess(X_tr, X_te):
    scaler = StandardScaler()
    return scaler.fit_transform(X_tr), scaler.transform(X_te)

output_file = "results/row_removal_diabetes.xlsx"
n_iter = 10
epsilon_values = [0.1, 1, 5, 10, 30, 50, 100]


### Baseline (full data, no removal, no perturbation)

In [4]:
X_train_sc, X_test_sc = preprocess(X_train, X_test)

baseline_model = LogisticRegression(
    penalty="l2",
    C=1,
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_train_sc, y_train)

print("Baseline (full data):")
fc.predict_binary(baseline_model, X_train_sc, y_train, conf_matrix=False)

fc.predict_binary_save_results(
    baseline_model,
    X_test_sc,
    y_test,
    conf_matrix=False,
    perturbation_type="Original",
    epsilon=0,
    row_id=None,
    output_file=output_file
)


Baseline (full data):
---------------------------------------
Accuracy: 0.7801
Precision: 0.7241
Recall: 0.5915
F1 Score: 0.6512
---------------------------------------
---------------------------------------
Accuracy: 0.7662
Precision: 0.6863
Recall: 0.6364
F1 Score: 0.6604
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx


----------------------------------------------------------------------------------------------
-------------------------------- INPUT PERTURBATION (row removal) ----------------------------
----------------------------------------------------------------------------------------------

In [5]:
ranges = [[0,20], [40,300], [30,200], [5,100], [2,900], [10,70], [0,3], [21,100]]
input_sensitivity = []
for r in ranges:
    input_sensitivity.append(r[1] - r[0])


In [6]:
row_rng = np.random.default_rng(42)   
np.random.seed(42)                    
for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    for e in epsilon_values:
        X_train_perturbed = X_train_reduced.copy()

        for i, var in enumerate(X_train_reduced.columns):
            eps_j = e  # The paper uses eps_j = e/d, but this leads to huge errors
            scale = input_sensitivity[i] / eps_j
            noise = np.random.laplace(loc=0, scale=scale, size=len(X_train_reduced))

            lower, upper = ranges[i]
            X_train_perturbed[var] = (
                X_train_reduced[var] + noise
            ).clip(lower=lower, upper=upper)

        X_train_perturbed_sc, X_test_perturbed_sc = preprocess(X_train_perturbed, X_test)

        input_model = copy.deepcopy(baseline_model)
        input_model.fit(X_train_perturbed_sc, y_train_reduced)

        print(f"Input - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_binary(input_model, X_train_perturbed_sc, y_train_reduced, conf_matrix=False)

        fc.predict_binary_save_results(
            input_model,
            X_test_perturbed_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Input",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Input - iteration 1/10, epsilon=0.1 - removed row 54
---------------------------------------
Accuracy: 0.6591
Precision: 0.8000
Recall: 0.0189
F1 Score: 0.0369
---------------------------------------
---------------------------------------
Accuracy: 0.6429
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 1/10, epsilon=1 - removed row 54
---------------------------------------
Accuracy: 0.6542
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
---------------------------------------
Accuracy: 0.6429
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 1/10, epsilon=5 - removed row 54
---------------------------------------
Accuracy: 0.6786
Precision: 0.5743
Recall: 0.2736
F1 Score: 0.3706
---------------------------------------
--------

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to results/row_removal_diabetes.xlsx
Input - iteration 2/10, epsilon=30 - removed row 475
---------------------------------------
Accuracy: 0.7667
Precision: 0.7143
Recall: 0.5425
F1 Score: 0.6166
---------------------------------------
---------------------------------------
Accuracy: 0.7727
Precision: 0.7083
Recall: 0.6182
F1 Score: 0.6602
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 2/10, epsilon=50 - removed row 475
---------------------------------------
Accuracy: 0.7618
Precision: 0.6941
Recall: 0.5566
F1 Score: 0.6178
---------------------------------------
---------------------------------------
Accuracy: 0.7597
Precision: 0.6731
Recall: 0.6364
F1 Score: 0.6542
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 2/10, epsilon=100 - removed row 475
---------------------------------------
Accuracy: 0.7716
Precision: 0.7093
Recall: 0.5755
F1 Score:

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to results/row_removal_diabetes.xlsx
Input - iteration 4/10, epsilon=5 - removed row 269
---------------------------------------
Accuracy: 0.6737
Precision: 0.5657
Recall: 0.2629
F1 Score: 0.3590
---------------------------------------
---------------------------------------
Accuracy: 0.6753
Precision: 1.0000
Recall: 0.0909
F1 Score: 0.1667
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 4/10, epsilon=10 - removed row 269
---------------------------------------
Accuracy: 0.7390
Precision: 0.6646
Recall: 0.5023
F1 Score: 0.5722
---------------------------------------
---------------------------------------
Accuracy: 0.7662
Precision: 0.7317
Recall: 0.5455
F1 Score: 0.6250
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 4/10, epsilon=30 - removed row 269
---------------------------------------
Accuracy: 0.7618
Precision: 0.6959
Recall: 0.5587
F1 Score: 0

...

Saved results to results/row_removal_diabetes.xlsx
Input - iteration 5/10, epsilon=100 - removed row 265
---------------------------------------
Accuracy: 0.7651
Precision: 0.6949
Recall: 0.5775
F1 Score: 0.6308
---------------------------------------
---------------------------------------
Accuracy: 0.7532
Precision: 0.6604
Recall: 0.6364
F1 Score: 0.6481
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 6/10, epsilon=0.1 - removed row 527
---------------------------------------
Accuracy: 0.6542
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
---------------------------------------
Accuracy: 0.6429
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 6/10, epsilon=1 - removed row 527
---------------------------------------
Accuracy: 0.6493
Precision: 0.4400
Recall: 0.0519
F1 Score:

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to results/row_removal_diabetes.xlsx
Input - iteration 7/10, epsilon=10 - removed row 52
---------------------------------------
Accuracy: 0.7096
Precision: 0.6207
Recall: 0.4225
F1 Score: 0.5028
---------------------------------------
---------------------------------------
Accuracy: 0.7532
Precision: 0.7429
Recall: 0.4727
F1 Score: 0.5778
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 7/10, epsilon=30 - removed row 52
---------------------------------------
Accuracy: 0.7700
Precision: 0.7045
Recall: 0.5822
F1 Score: 0.6375
---------------------------------------
---------------------------------------
Accuracy: 0.7532
Precision: 0.6735
Recall: 0.6000
F1 Score: 0.6346
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 7/10, epsilon=50 - removed row 52
---------------------------------------
Accuracy: 0.7667
Precision: 0.7108
Recall: 0.5540
F1 Score: 0.6

/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/elif/Desktop/Projects/privacy_in_ml/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Saved results to results/row_removal_diabetes.xlsx
Input - iteration 10/10, epsilon=1 - removed row 57
---------------------------------------
Accuracy: 0.6525
Precision: 0.5000
Recall: 0.0141
F1 Score: 0.0274
---------------------------------------
---------------------------------------
Accuracy: 0.6429
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 10/10, epsilon=5 - removed row 57
---------------------------------------
Accuracy: 0.6705
Precision: 0.5733
Recall: 0.2019
F1 Score: 0.2986
---------------------------------------
---------------------------------------
Accuracy: 0.6494
Precision: 0.6667
Recall: 0.0364
F1 Score: 0.0690
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Input - iteration 10/10, epsilon=10 - removed row 57
---------------------------------------
Accuracy: 0.7423
Precision: 0.6774
Recall: 0.4930
F1 Score: 0.

----------------------------------------------------------------------------------------------
-------------------------------- OUTPUT PERTURBATION (row removal) ---------------------------
----------------------------------------------------------------------------------------------

In [7]:
row_rng = np.random.default_rng(42)   # reset: same 10 rows as the Input section above
np.random.seed(42)

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    reduced_model = copy.deepcopy(baseline_model)
    reduced_model.fit(X_train_reduced_sc, y_train_reduced)

    n = X_train_reduced.shape[0]
    C_x = np.max(np.linalg.norm(X_train_reduced_sc, axis=1))
    lambda_reg = 1 / reduced_model.C
    output_sensitivity = (C_x * 2) / (lambda_reg * n)

    for e in epsilon_values:
        coef_noise = np.random.laplace(loc=0, scale=output_sensitivity / e, size=reduced_model.coef_.shape)
        intercept_noise = np.random.laplace(loc=0, scale=output_sensitivity / e, size=reduced_model.intercept_.shape)

        output_model = copy.deepcopy(reduced_model)
        output_model.coef_ = reduced_model.coef_ + coef_noise
        output_model.intercept_ = reduced_model.intercept_ + intercept_noise

        print(f"Output - iteration {iteration}/{n_iter}, epsilon={e} - removed row {removed_label}")
        fc.predict_binary(output_model, X_train_reduced_sc, y_train_reduced, conf_matrix=False)

        fc.predict_binary_save_results(
            output_model,
            X_test_reduced_sc,
            y_test,
            conf_matrix=False,
            perturbation_type="Output",
            epsilon=e,
            row_id=removed_label,
            output_file=output_file
        )


Output - iteration 1/10, epsilon=0.1 - removed row 54
---------------------------------------
Accuracy: 0.7471
Precision: 0.6462
Recall: 0.5943
F1 Score: 0.6192
---------------------------------------
---------------------------------------
Accuracy: 0.7143
Precision: 0.6038
Recall: 0.5818
F1 Score: 0.5926
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Output - iteration 1/10, epsilon=1 - removed row 54
---------------------------------------
Accuracy: 0.7765
Precision: 0.7219
Recall: 0.5755
F1 Score: 0.6404
---------------------------------------
---------------------------------------
Accuracy: 0.7662
Precision: 0.6863
Recall: 0.6364
F1 Score: 0.6604
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Output - iteration 1/10, epsilon=5 - removed row 54
---------------------------------------
Accuracy: 0.7798
Precision: 0.7225
Recall: 0.5896
F1 Score: 0.6494
---------------------------------------
-----

----------------------------------------------------------------------------------------------
-------------------------------- INTERNAL PERTURBATION (row removal) -------------------------
----------------------------------------------------------------------------------------------

In [8]:
row_rng = np.random.default_rng(42)   # reset: same 10 rows as Input/Output above

for iteration in range(1, n_iter + 1):
    idx = row_rng.integers(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    data_norm = np.max(np.linalg.norm(X_train_reduced_sc, axis=1))

    print(f"Internal - iteration {iteration}/{n_iter} - removed row {removed_label}, data_norm={data_norm:.4f}")

    fc.internal_perturbation_save_results(
        "diabetes_row_removal",
        X_train_reduced_sc,
        y_train_reduced,
        X_test_reduced_sc,
        y_test,
        epsilon_values=epsilon_values,
        data_norm=data_norm,
        C=baseline_model.C,
        bivariate=True,
        row_id=removed_label,
        output_file=output_file,
        perturbation_type="Internal"
    )


Internal - iteration 1/10 - removed row 54, data_norm=8.6105
---------------------------------------
Epsilon: 0.1
Accuracy: 0.5195
Precision: 0.3855
Recall: 0.5818
F1 Score: 0.4638
Time: 0.00 s
---------------------------------------
---------------------------------------
Epsilon: 1
Accuracy: 0.6169
Precision: 0.4722
Recall: 0.6182
F1 Score: 0.5354
Time: 0.00 s
---------------------------------------
---------------------------------------
Epsilon: 5
Accuracy: 0.7403
Precision: 0.6271
Recall: 0.6727
F1 Score: 0.6491
Time: 0.00 s
---------------------------------------
---------------------------------------
Epsilon: 10
Accuracy: 0.7468
Precision: 0.6481
Recall: 0.6364
F1 Score: 0.6422
Time: 0.00 s
---------------------------------------
---------------------------------------
Epsilon: 30
Accuracy: 0.7532
Precision: 0.6604
Recall: 0.6364
F1 Score: 0.6481
Time: 0.00 s
---------------------------------------
---------------------------------------
Epsilon: 50
Accuracy: 0.7532
Precision: 